In [2]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
REPO_DIR = Path('/content/Yolo-ST-GCN')
BRANCH = 'refactor-1'

# Install lightweight dependencies needed for smoke checks.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'torch', '-q'],
    check=True,
)

# Clone or update exact branch used for current development.
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', os.getcwd())
print('Branch:', BRANCH)

Repo ready: /content/Yolo-ST-GCN
Branch: refactor-1


In [3]:
import importlib
import os
import tempfile

import numpy as np
import torch

print('=== Refactor Smoke Test (JointSpec + Config-driven) ===')

required_modules = [
    'src.joint_specs',
    'src.checkpointing',
    'src.experiment_config',
]
missing = []
for m in required_modules:
    try:
        importlib.import_module(m)
    except Exception:
        missing.append(m)

if missing:
    print('Refactor code chua co day du tren branch hien tai.')
    print('Missing modules:')
    for m in missing:
        print(' -', m)
    print('\nHay dam bao da push code moi len branch "duy", sau do chay lai Cell 1 de git pull roi test lai Cell 2.')
else:
    from src.checkpointing import load_checkpoint, save_checkpoint
    from src.experiment_config import apply_overrides
    from src.joint_specs import get_joint_spec
    from src.model import Model_STGCN
    from src.skeleton_utils import to_stgcn_input_from_coco17_with_spec

    # 1) JointSpec registry check
    spec14 = get_joint_spec('penn14')
    spec18 = get_joint_spec('coco18')
    print('Spec penn14:', {'num_joints': spec14.num_joints, 'center_joint': spec14.center_joint})
    print('Spec coco18:', {'num_joints': spec18.num_joints, 'center_joint': spec18.center_joint})

    assert spec14.num_joints == 14
    assert spec18.num_joints == 18

    # 2) COCO17 -> penn14/coco18 conversion check
    coco_seq = np.random.rand(48, 17, 2).astype(np.float32)
    x14 = to_stgcn_input_from_coco17_with_spec(coco_seq, 'penn14', target_frames=64)
    x18 = to_stgcn_input_from_coco17_with_spec(coco_seq, 'coco18', target_frames=64)
    print('Converted tensor penn14 shape:', x14.shape)
    print('Converted tensor coco18 shape:', x18.shape)

    assert x14.shape == (2, 64, 14, 1)
    assert x18.shape == (2, 64, 18, 1)

    # 3) Dynamic model graph check + forward check
    model14 = Model_STGCN(num_classes=99, joint_spec='penn14').eval()
    model18 = Model_STGCN(num_classes=99, joint_spec='coco18').eval()

    with torch.no_grad():
        out14 = model14(torch.from_numpy(np.expand_dims(x14, axis=0)))
        out18 = model18(torch.from_numpy(np.expand_dims(x18, axis=0)))

    print('Model penn14 graph nodes:', model14.graph.num_node, 'output:', tuple(out14.shape))
    print('Model coco18 graph nodes:', model18.graph.num_node, 'output:', tuple(out18.shape))

    assert tuple(out14.shape) == (1, 99)
    assert tuple(out18.shape) == (1, 99)

    # 4) Checkpoint metadata round-trip
    with tempfile.TemporaryDirectory(prefix='stgcn_ckpt_test_') as td:
        ckpt_path = os.path.join(td, 'model_test.pth')
        meta_in = {
            'joint_spec_name': 'coco18',
            'use_two_stream': True,
            'dataset_format': 'gym99',
            'num_classes': 99,
        }
        save_checkpoint(ckpt_path, model18, meta_in)
        state_dict, meta_out = load_checkpoint(ckpt_path, map_location='cpu')

        print('Checkpoint metadata out:', meta_out)
        assert 'fcn.weight' in state_dict
        assert meta_out['joint_spec_name'] == 'coco18'
        assert meta_out['use_two_stream'] is True

    # 5) Config override behavior (CLI wins)
    class ArgsObj:
        def __init__(self):
            self.joint_spec_name = 'penn14'
            self.num_workers = 0
            self.epochs = 30

    args_obj = ArgsObj()
    config_dict = {'joint_spec_name': 'coco18', 'num_workers': 2, 'epochs': 50}
    args_obj = apply_overrides(args_obj, config_dict, cli_tokens=['--epochs', '30'])

    print('Config override result:', {
        'joint_spec_name': args_obj.joint_spec_name,
        'num_workers': args_obj.num_workers,
        'epochs': args_obj.epochs,
    })

    assert args_obj.joint_spec_name == 'coco18'
    assert args_obj.num_workers == 2
    assert args_obj.epochs == 30

    print('\nALL TESTS PASSED')

=== Refactor Smoke Test (JointSpec + Config-driven) ===
Spec penn14: {'num_joints': 14, 'center_joint': 13}
Spec coco18: {'num_joints': 18, 'center_joint': 17}
Converted tensor penn14 shape: (2, 64, 14, 1)
Converted tensor coco18 shape: (2, 64, 18, 1)
Model penn14 graph nodes: 14 output: (1, 99)
Model coco18 graph nodes: 18 output: (1, 99)
Checkpoint metadata out: {'joint_spec_name': 'coco18', 'use_two_stream': True, 'dataset_format': 'gym99', 'num_classes': 99}
Config override result: {'joint_spec_name': 'coco18', 'num_workers': 2, 'epochs': 30}

ALL TESTS PASSED


In [ ]:
# !git pull

In [4]:
import os
import pickle
import subprocess
import sys
import urllib.request
from pathlib import Path

print('=== 2-Epoch Train Matrix Test: Gym288 + Gym99 ===')

# ---------------------------------------------------------------------
# 0) Runtime hotfixes for Colab branch drift
# ---------------------------------------------------------------------
# Hotfix A: old import in two_stream_stgcn.py referencing removed symbol.
two_stream_module = Path('src/two_stream_stgcn.py')
if two_stream_module.exists():
    text = two_stream_module.read_text(encoding='utf-8')
    old_import = 'from src.model import Model_STGCN, Model_STGCN_COCO18'
    new_import = 'from src.model import Model_STGCN'
    if old_import in text:
        text = text.replace(old_import, new_import)
        two_stream_module.write_text(text, encoding='utf-8')
        print('Applied runtime hotfix in src/two_stream_stgcn.py')

# Hotfix B: invalid default in gym99_dataset.py.
gym99_module = Path('src/gym99_dataset.py')
if gym99_module.exists():
    text = gym99_module.read_text(encoding='utf-8')
    if 'COCO17_BONE_PAIRS_18' in text:
        text = text.replace('COCO17_BONE_PAIRS_18', 'PENN_BONE_PAIRS_14')
        gym99_module.write_text(text, encoding='utf-8')
        print('Applied runtime hotfix in src/gym99_dataset.py')

# ---------------------------------------------------------------------
# 1) Ensure dataset tools are available
# ---------------------------------------------------------------------
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from huggingface_hub import snapshot_download

# ---------------------------------------------------------------------
# 2) Ensure Gym288 dataset exists
# ---------------------------------------------------------------------
download_dir = Path('/content/Gym288-skeleton')
if not (download_dir.exists() and any(download_dir.iterdir())):
    print('Downloading Gym288-skeleton...')
    download_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(download_dir),
        local_dir_use_symlinks=False,
    )
else:
    print('Gym288-skeleton already exists.')

pkl_candidates = sorted(download_dir.rglob('*.pkl'))
if not pkl_candidates:
    raise FileNotFoundError('No .pkl file found for Gym288 dataset.')
GYM288_DATASET_PATH = pkl_candidates[0]
print('Gym288 pickle:', GYM288_DATASET_PATH)

# ---------------------------------------------------------------------
# 3) Build Gym99-from-Gym288 pickle (robust parser, no regex escaping risk)
# ---------------------------------------------------------------------
GYM99_DIR = Path('/content/Gym99-from-Gym288')
GYM99_DIR.mkdir(parents=True, exist_ok=True)
GYM99_DATASET_PATH = GYM99_DIR / 'gym99_from_gym288.pkl'

print('Building/refreshing Gym99-from-Gym288 pickle...')


def parse_finegym_categories(text: str):
    pairs = []
    for line in text.splitlines():
        if 'Clabel:' not in line or 'Glabel:' not in line:
            continue
        try:
            clabel = int(line.split('Clabel:')[1].split(';')[0].strip())
            glabel = int(line.split('Glabel:')[1].split(';')[0].strip())
            pairs.append((clabel, glabel))
        except Exception:
            continue
    return pairs


url_288 = 'https://sdolivia.github.io/FineGym/resources/dataset/gym288_categories.txt'
url_99 = 'https://sdolivia.github.io/FineGym/resources/dataset/gym99_categories.txt'
text_288 = urllib.request.urlopen(url_288).read().decode('utf-8')
text_99 = urllib.request.urlopen(url_99).read().decode('utf-8')

pairs_288 = parse_finegym_categories(text_288)
pairs_99 = parse_finegym_categories(text_99)

if not pairs_288 or not pairs_99:
    raise RuntimeError('Cannot parse FineGym category files for gym288/gym99.')

clabel288_to_glabel = dict(pairs_288)
glabel_to_clabel99 = {g: c for c, g in pairs_99}

map_288_to_99 = {}
for clabel_288, glabel in clabel288_to_glabel.items():
    if glabel in glabel_to_clabel99:
        map_288_to_99[clabel_288] = glabel_to_clabel99[glabel]

with open(str(GYM288_DATASET_PATH), 'rb') as f:
    gym288_payload = pickle.load(f)

annotations = gym288_payload.get('annotations', [])
split_info = gym288_payload.get('split', {})

gym99_annotations = []
valid_ids = set()
matched_direct = 0
matched_minus1 = 0
matched_plus1 = 0

for ann in annotations:
    lbl = int(ann['label'])

    mapped = None
    if lbl in map_288_to_99:
        mapped = map_288_to_99[lbl]
        matched_direct += 1
    elif (lbl - 1) in map_288_to_99:
        mapped = map_288_to_99[lbl - 1]
        matched_minus1 += 1
    elif (lbl + 1) in map_288_to_99:
        mapped = map_288_to_99[lbl + 1]
        matched_plus1 += 1

    if mapped is None:
        continue

    ann_new = dict(ann)
    ann_new['label'] = int(mapped)
    gym99_annotations.append(ann_new)
    valid_ids.add(str(ann_new['frame_dir']))

gym99_split = {
    'train': [vid for vid in split_info.get('train', []) if vid in valid_ids],
    'test': [vid for vid in split_info.get('test', []) if vid in valid_ids],
}

gym99_payload = {
    'split': gym99_split,
    'annotations': gym99_annotations,
}

with open(str(GYM99_DATASET_PATH), 'wb') as f:
    pickle.dump(gym99_payload, f)

print('Gym99 pickle:', GYM99_DATASET_PATH)
print('Gym99 matches: direct=', matched_direct, ' minus1=', matched_minus1, ' plus1=', matched_plus1)
print('Gym99 train/test:', len(gym99_split['train']), len(gym99_split['test']))

if len(gym99_split['train']) == 0 or len(gym99_split['test']) == 0:
    raise RuntimeError('Gym99 split is empty after mapping; stop before train.')

# ---------------------------------------------------------------------
# 4) Run training matrix: 2 epochs each
# ---------------------------------------------------------------------
run_plan = [
    ('gym288', 'penn14'),
    ('gym288', 'coco18'),
    ('gym99', 'penn14'),
    ('gym99', 'coco18'),
]

for dataset_name, joint_spec in run_plan:
    out_dir = Path(f'outputs/refactor1_2ep/{dataset_name}_{joint_spec}_2s')
    out_dir.mkdir(parents=True, exist_ok=True)

    if dataset_name == 'gym288':
        cmd = [
            sys.executable, 'scripts/train_gym288.py',
            '--dataset_path', str(GYM288_DATASET_PATH),
            '--out_dir', str(out_dir),
            '--epochs', '2',
            '--batch_size', '256',
            '--lr', '0.001',
            '--joint_spec_name', joint_spec,
            '--use_two_stream',
        ]
    else:
        cmd = [
            sys.executable, 'scripts/train_gym99.py',
            '--dataset_path', str(GYM99_DATASET_PATH),
            '--out_dir', str(out_dir),
            '--epochs', '2',
            '--batch_size', '512',
            '--lr', '0.001',
            '--num_workers', '2',
            '--joint_spec_name', joint_spec,
            '--use_two_stream',
        ]

    print('\n' + '=' * 90)
    print(f'RUN: {dataset_name} | joint_spec={joint_spec}')
    print('CMD:', ' '.join(cmd))
    print('=' * 90)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')

    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

    print(f'Completed: {dataset_name} | {joint_spec}')
    print('Artifacts in:', out_dir)

print('\nALL 2-EPOCH TRAIN RUNS COMPLETED')

=== 2-Epoch Train Matrix Test: Gym288 + Gym99 ===
Applied runtime hotfix in src/two_stream_stgcn.py
Gym288-skeleton already exists.
Gym288 pickle: /content/Gym288-skeleton/gym_288_skeleton.pkl
Building/refreshing Gym99-from-Gym288 pickle...
Gym99 pickle: /content/Gym99-from-Gym288/gym99_from_gym288.pkl
Gym99 matches: direct= 34240  minus1= 1625  plus1= 800
Gym99 train/test: 27624 9041

RUN: gym288 | joint_spec=penn14
CMD: /usr/bin/python3 scripts/train_gym288.py --dataset_path /content/Gym288-skeleton/gym_288_skeleton.pkl --out_dir outputs/refactor1_2ep/gym288_penn14_2s --epochs 2 --batch_size 256 --lr 0.001 --joint_spec_name penn14 --use_two_stream
Device: cuda
Loading Gym288-skeleton dataset...
Loaded 38223 samples  train=28739  test=9484
num_classes=288 (inferred=288)
DataLoader num_workers=0  two_stream=True

Epoch 1/2 [train]: 100%|██████████| 113/113 [03:23<00:00,  1.45s/it]
                                                                    

Epoch 1/2 [val]: 100%|██████████| 38

KeyboardInterrupt: 

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'scripts/train_gym288.py',
    '--dataset_path', '/content/Gym288-skeleton/gym_288_skeleton.pkl',
    '--out_dir', 'outputs/refactor1_2ep/gym288_coco18_2s',
    '--epochs', '2',
    '--batch_size', '256',
    '--lr', '0.001',
    '--joint_spec_name', 'coco18',
    '--use_two_stream',
]

print('Running diagnostic command:')
print(' '.join(cmd))

proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Return code:', proc.returncode)

lines = proc.stdout.splitlines()
print('\n--- Last 120 log lines ---')
for ln in lines[-120:]:
    print(ln)

if proc.returncode != 0:
    raise RuntimeError('Diagnostic run failed; check log above.')